# 🏍️ BikeWale — Data Collection, Cleaning & EDA Project

| | |
|---|---|
| **Website** | https://www.bikewale.com |
| **Topic** | New & Used Bikes in India |
| **Target Feature** | Price (INR) |
| **Other Features** | Brand, Model, Engine CC, Mileage, Weight, Fuel Tank |

---
## 📋 Problem Statement
> To analyze the factors affecting bike prices in India — such as brand, engine displacement, mileage, and weight — and understand pricing trends across different bike segments using data scraped from BikeWale.com.


---
## Step 1 — Import Libraries

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import re, json, time
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)

print("✅ All libraries imported successfully")

---
## Step 2 — Web Scraping (BikeWale)
> Scraping bike data from BikeWale using `requests` and `BeautifulSoup`

In [ ]:
# Connect to BikeWale
url = "https://www.bikewale.com/new-bikes-in-india/"
page = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})

print("Status Code:", page.status_code)
print("Page Size:", len(page.text), "characters")

In [ ]:
# Parse HTML with BeautifulSoup
soup = BeautifulSoup(page.text, "html.parser")

# Verify bike data exists in HTML
for keyword in ["Pulsar", "Splendor", "price"]:
    if keyword.lower() in page.text.lower():
        print(f"✅ '{keyword}' found in HTML!")

In [ ]:
# Extract all individual bike page URLs using Regex
bike_urls = re.findall(
    r'href=["\'](/[a-z-]+-bikes/[a-z0-9-]+/)["\']',
    page.text
)
bike_urls = list(set(bike_urls))
print(f"Total bike URLs found from homepage: {len(bike_urls)}")
for url in bike_urls[:10]:
    print("https://www.bikewale.com" + url)

In [ ]:
# Test scraping one bike page - Check JSON structured data
test_url = "https://www.bikewale.com/bajaj-bikes/platina-110/"
test_page = requests.get(test_url, headers={"User-Agent": "Mozilla/5.0"})
test_soup = BeautifulSoup(test_page.text, "html.parser")

scripts = test_soup.find_all("script", type="application/ld+json")
print(f"JSON scripts found: {len(scripts)}")
for i, s in enumerate(scripts):
    try:
        data = json.loads(s.string)
        if data.get("@type") == "Product":
            print(f"\n✅ Product data found in Script {i}:")
            print(json.dumps(data, indent=2)[:400])
    except:
        pass

In [ ]:
# Define scraping function to extract all bike specs
def scrape_bike(url):
    try:
        page = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
        soup = BeautifulSoup(page.text, "html.parser")
        scripts = soup.find_all("script", type="application/ld+json")

        description = ""
        for s in scripts:
            try:
                data = json.loads(s.string)
                if data.get("@type") == "Product":
                    description = data.get("description", "")
                    break
            except:
                pass

        # Extract brand and model from URL
        url_parts = url.strip("/").split("/")
        brand_part = url_parts[-2].replace("-bikes", "").replace("-", " ").title()
        model_part = url_parts[-1].replace("-", " ").title()

        # Extract features using Regex
        price   = re.findall(r'Rs\.\s*([\d,]+)', description)
        engine  = re.findall(r'([\d.]+)\s*cc', description)
        mileage = re.findall(r'mileage of ([\d.]+)\s*kmpl', description)
        weight  = re.findall(r'weighs ([\d.]+)\s*kg', description)
        tank    = re.findall(r'fuel tank capa[a-z]* of ([\d.]+)', description)

        return {
            "Brand":        brand_part,
            "Model":        model_part,
            "Price_INR":    price[0].replace(",", "") if price else np.nan,
            "Engine_CC":    engine[0] if engine else np.nan,
            "Mileage_KMPL": mileage[0] if mileage else np.nan,
            "Weight_KG":    weight[0] if weight else np.nan,
            "Fuel_Tank_L":  tank[0] if tank else np.nan,
        }
    except Exception as e:
        return None

# Test the function
result = scrape_bike("https://www.bikewale.com/bajaj-bikes/platina-110/")
print("Test Result:", result)

In [ ]:
# Collect URLs from all major brand pages
more_urls = []
brands = [
    "bajaj", "hero", "honda", "tvs", "yamaha",
    "royalenfield", "ktm", "suzuki", "kawasaki",
    "ather", "ola", "jawa", "aprilia", "triumph",
    "harley-davidson", "bmw", "ducati", "benelli",
    "vespa", "revolt"
]

for brand in brands:
    url = f"https://www.bikewale.com/{brand}-bikes/"
    try:
        pg = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
        found = re.findall(
            r'href=["\'](/' + brand + r'-bikes/[a-z0-9-]+/)["\']', pg.text
        )
        found = list(set(found))
        more_urls.extend(found)
        print(f"✅ {brand}: {len(found)} bikes found")
        time.sleep(1)
    except Exception as e:
        print(f"❌ {brand}: {e}")

all_urls = list(set(bike_urls + more_urls))
print(f"\n✅ Total unique bike URLs: {len(all_urls)}")

In [ ]:
# Scrape first 450 bikes (above 400 minimum requirement)
all_bikes = []
urls_to_scrape = all_urls[:450]

print(f"Scraping {len(urls_to_scrape)} bikes...\n")
for i, url in enumerate(urls_to_scrape):
    full_url = "https://www.bikewale.com" + url
    result = scrape_bike(full_url)
    if result:
        all_bikes.append(result)
        print(f"✅ {i+1}/{len(urls_to_scrape)} → {result['Brand']} {result['Model']}")
    time.sleep(0.5)

print(f"\n✅ Total bikes scraped: {len(all_bikes)}")

---
## Step 3 — Create DataFrame
> Convert scraped data into a Pandas DataFrame

In [ ]:
import pandas as pd

df = pd.DataFrame(all_bikes)
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
df.head(10)

---
## Step 4 — Export to CSV
> Save the DataFrame as a CSV file

In [ ]:
df.to_csv("bikewale_data.csv", index=False)
print("✅ CSV saved successfully as 'bikewale_data.csv'")

---
## Step 5 — Read CSV & Inspect Data
> Load the CSV and understand its structure

In [ ]:
df = pd.read_csv("bikewale_data.csv")

print("=" * 40)
print("Shape (Rows, Columns):", df.shape)
print("=" * 40)
print("\nColumn Names:", df.columns.tolist())
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Statistical Summary
print("Statistical Description:")
df.describe()

---
## Step 6 — Data Cleaning
> Cleaning steps:
> - Convert data types
> - Handle missing values
> - Remove duplicates
> - Treat outliers

In [ ]:
# Check before cleaning
print("Shape before cleaning:", df.shape)
print("Duplicates:", df.duplicated().sum())

In [ ]:
# Remove duplicates
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print("Shape after removing duplicates:", df.shape)

In [ ]:
# Convert columns to numeric data types
numeric_cols = ['Price_INR', 'Engine_CC', 'Mileage_KMPL', 'Weight_KG', 'Fuel_Tank_L']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("Data types after conversion:")
print(df.dtypes)

In [ ]:
# Handle missing values
print("Missing values before treatment:")
print(df.isnull().sum())

# Fill numerical missing values with median
for col in numeric_cols:
    median_val = df[col].median()
    df[col].fillna(median_val, inplace=True)
    print(f"  {col}: filled with median = {median_val}")

print("\nMissing values after treatment:")
print(df.isnull().sum())

In [ ]:
# Standardise text columns
df['Brand'] = df['Brand'].str.strip().str.title()
df['Model'] = df['Model'].str.strip().str.title()

# Fix brand casing for acronyms (title() wrongly converts KTM→Ktm, TVS→Tvs, BMW→Bmw)
uppercase_brands = {'Ktm': 'KTM', 'Tvs': 'TVS', 'Bmw': 'BMW'}
df['Brand'] = df['Brand'].replace(uppercase_brands)
print("Brands after fix:", sorted(df['Brand'].unique()))


# Remove invalid prices (too low or zero)
print("Price range before:", df['Price_INR'].min(), "to", df['Price_INR'].max())
df = df[df['Price_INR'] > 10000]
print("Price range after:", df['Price_INR'].min(), "to", df['Price_INR'].max())

In [ ]:
# Treat Outliers using IQR method on Price
Q1 = df['Price_INR'].quantile(0.25)
Q3 = df['Price_INR'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df['Price_INR'] < lower) | (df['Price_INR'] > upper)]
print(f"Outliers found: {len(outliers)}")
print(f"IQR Range: ₹{lower:,.0f} to ₹{upper:,.0f}")

# Keep outliers but flag them
df['Is_Outlier'] = ((df['Price_INR'] < lower) | (df['Price_INR'] > upper))
print("\nFinal cleaned shape:", df.shape)
df.head()

---
## Step 7 — Univariate Analysis
> Analyzing each variable individually
### Numerical Variables: Central Tendency & Dispersion
### Categorical Variables: Frequency & Distribution

In [ ]:
# Central Tendency for Numerical Columns
print("=" * 50)
print("CENTRAL TENDENCY & DISPERSION")
print("=" * 50)
for col in ['Price_INR', 'Engine_CC', 'Mileage_KMPL', 'Weight_KG', 'Fuel_Tank_L']:
    print(f"\n{col}:")
    print(f"  Mean   : {df[col].mean():.2f}")
    print(f"  Median : {df[col].median():.2f}")
    print(f"  Mode   : {df[col].mode()[0]:.2f}")
    print(f"  Std Dev: {df[col].std():.2f}")
    print(f"  Min    : {df[col].min():.2f}")
    print(f"  Max    : {df[col].max():.2f}")

In [ ]:
# Price Distribution — Histogram, KDE, Boxplot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Price Distribution (INR)', fontsize=14, fontweight='bold')

axes[0].hist(df['Price_INR'], bins=40, color='#3498db', edgecolor='white')
axes[0].set_title('Histogram')
axes[0].set_xlabel('Price (INR)')
axes[0].set_ylabel('Frequency')

sns.kdeplot(df['Price_INR'], ax=axes[1], fill=True, color='#e74c3c')
axes[1].set_title('KDE / Distribution Plot')
axes[1].set_xlabel('Price (INR)')

sns.boxplot(y=df['Price_INR'], ax=axes[2], color='#2ecc71')
axes[2].set_title('Boxplot')

plt.tight_layout()
plt.show()
print("Interpretation: Price is right-skewed. Most bikes are priced between ₹50K-₹3L. High-end superbikes are outliers.")

In [ ]:
# Mileage Distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Mileage Distribution (KMPL)', fontsize=14, fontweight='bold')

axes[0].hist(df['Mileage_KMPL'], bins=30, color='#27ae60', edgecolor='white')
axes[0].set_title('Histogram')
axes[0].set_xlabel('Mileage (KMPL)')

sns.kdeplot(df['Mileage_KMPL'], ax=axes[1], fill=True, color='#8e44ad')
axes[1].set_title('KDE Plot')

sns.violinplot(y=df['Mileage_KMPL'], ax=axes[2], color='#f39c12')
axes[2].set_title('Violin Plot')

plt.tight_layout()
plt.show()
print("Interpretation: Most bikes give 40-70 kmpl mileage. Commuter bikes have higher mileage than sports bikes.")

In [ ]:
# Engine CC Distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Engine CC Distribution', fontsize=14, fontweight='bold')

axes[0].hist(df['Engine_CC'], bins=30, color='#e67e22', edgecolor='white')
axes[0].set_title('Histogram')
axes[0].set_xlabel('Engine CC')

sns.kdeplot(df['Engine_CC'], ax=axes[1], fill=True, color='#2980b9')
axes[1].set_title('KDE Plot')

sns.boxplot(y=df['Engine_CC'], ax=axes[2], color='#c0392b')
axes[2].set_title('Boxplot')

plt.tight_layout()
plt.show()
print("Interpretation: Most bikes have 100-200cc engines. High CC bikes (400cc+) are premium segment.")

In [ ]:
# Categorical — Brand Count
fig, ax = plt.subplots(figsize=(14, 6))
brand_counts = df['Brand'].value_counts()
sns.barplot(x=brand_counts.index, y=brand_counts.values, palette='tab20', ax=ax)
ax.set_title('Number of Bike Models per Brand', fontsize=13, fontweight='bold')
ax.set_xlabel('Brand')
ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
print("Interpretation: Kawasaki, Honda and Yamaha have the most models. Revolt and Ola are newer EV brands.")

In [ ]:
# Categorical — Frequency Table for Brand
print("Brand Frequency Table:")
brand_freq = df['Brand'].value_counts().reset_index()
brand_freq.columns = ['Brand', 'Count']
brand_freq['Percentage'] = (brand_freq['Count'] / len(df) * 100).round(2)
print(brand_freq.to_string(index=False))

In [ ]:
# Pie Chart — Top 8 Brands
top_brands = df['Brand'].value_counts().head(8)
fig, ax = plt.subplots(figsize=(9, 9))
ax.pie(top_brands, labels=top_brands.index, autopct='%1.1f%%',
       colors=sns.color_palette('Set2', 8), startangle=90)
ax.set_title('Top 8 Brands by Model Count', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("Interpretation: Top 3 brands (Kawasaki, Honda, Yamaha) together account for majority of bike models listed.")

---
## Step 8 — Bivariate & Multivariate Analysis
> - Continuous vs Continuous → Scatter plot, Correlation Heatmap
> - Continuous vs Categorical → Groupby, Pivot Table, Boxplot
> - Categorical vs Categorical → Crosstab

In [ ]:
# Scatter Plot — Engine CC vs Price
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(df['Engine_CC'], df['Price_INR'], alpha=0.4, color='#3498db', s=30)
axes[0].set_title('Engine CC vs Price', fontweight='bold')
axes[0].set_xlabel('Engine CC')
axes[0].set_ylabel('Price (INR)')

axes[1].scatter(df['Mileage_KMPL'], df['Price_INR'], alpha=0.4, color='#e74c3c', s=30)
axes[1].set_title('Mileage vs Price', fontweight='bold')
axes[1].set_xlabel('Mileage (KMPL)')
axes[1].set_ylabel('Price (INR)')

plt.tight_layout()
plt.show()
print("Interpretation: Higher engine CC strongly correlates with higher price. Mileage has inverse relation with price.")

In [ ]:
# Correlation Heatmap
num_cols = ['Price_INR', 'Engine_CC', 'Mileage_KMPL', 'Weight_KG', 'Fuel_Tank_L']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.5, linecolor='white', annot_kws={'size': 10}, ax=ax)
ax.set_title('Correlation Heatmap — Numerical Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("Interpretation: Engine CC and Weight have strong positive correlation with Price (r≈0.85). Mileage has negative correlation.")

In [ ]:
# Brand vs Average Price — GroupBy
print("=== GroupBy: Brand → Average Price ===")
gb = df.groupby('Brand')['Price_INR'].agg(['mean', 'min', 'max', 'count'])
gb.columns = ['Avg_Price', 'Min_Price', 'Max_Price', 'Count']
gb = gb.sort_values('Avg_Price', ascending=False)
print(gb.round(0).to_string())

In [ ]:
# Bar Chart — Brand vs Average Price
fig, ax = plt.subplots(figsize=(14, 7))
top_brands_price = gb.head(15)
bars = ax.barh(top_brands_price.index, top_brands_price['Avg_Price'],
               color=sns.color_palette('Spectral', len(top_brands_price)))
ax.set_title('Top 15 Brands by Average Price', fontsize=13, fontweight='bold')
ax.set_xlabel('Average Price (INR)')
for bar, val in zip(bars, top_brands_price['Avg_Price']):
    ax.text(val + 5000, bar.get_y() + bar.get_height()/2,
            f'₹{val/1e5:.1f}L', va='center', fontsize=8)
plt.tight_layout()
plt.show()
print("Interpretation: BMW, Ducati and Kawasaki are the most expensive brands. Hero and TVS are most affordable.")

In [ ]:
# Boxplot — Brand vs Price (Top 10 brands)
top10 = df['Brand'].value_counts().head(10).index
df_top10 = df[df['Brand'].isin(top10)]

fig, ax = plt.subplots(figsize=(14, 7))
sns.boxplot(data=df_top10, x='Brand', y='Price_INR', palette='Set3', ax=ax)
ax.set_title('Price Distribution by Brand (Top 10)', fontsize=13, fontweight='bold')
ax.set_xlabel('Brand')
ax.set_ylabel('Price (INR)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
print("Interpretation: Kawasaki shows widest price range. Hero and Bajaj have narrow, affordable price ranges.")

In [ ]:
# Pivot Table — Brand vs Average Mileage & Price
print("=== Pivot Table: Brand → Avg Price & Avg Mileage ===")
pivot = pd.pivot_table(
    df,
    values=['Price_INR', 'Mileage_KMPL', 'Engine_CC'],
    index='Brand',
    aggfunc='mean'
).round(1).sort_values('Price_INR', ascending=False)
print(pivot.to_string())

In [ ]:
# Crosstab — Brand vs Price Category
df['Price_Category'] = pd.cut(
    df['Price_INR'],
    bins=[0, 80000, 150000, 300000, 700000, float('inf')],
    labels=['Budget(<80K)', 'Mid(80K-1.5L)', 'Premium(1.5-3L)', 'High(3-7L)', 'Luxury(>7L)']
)

print("=== Crosstab: Brand vs Price Category ===")
ct = pd.crosstab(df['Brand'], df['Price_Category'])
print(ct.to_string())

In [ ]:
# Stacked Bar — Brand vs Price Category
ct_top10 = pd.crosstab(df[df['Brand'].isin(top10)]['Brand'],
                        df[df['Brand'].isin(top10)]['Price_Category'])
ct_top10.plot(kind='bar', stacked=True, figsize=(14, 7),
              colormap='Set2')
plt.title('Brand vs Price Category (Top 10 Brands)', fontsize=13, fontweight='bold')
plt.xlabel('Brand')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.legend(title='Price Category', bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()
print("Interpretation: Hero and Bajaj are mostly in Budget/Mid segment. Kawasaki and BMW are mostly Luxury.")

In [ ]:
# KDE Plot — Price by Top 5 Brands
top5 = df['Brand'].value_counts().head(5).index
fig, ax = plt.subplots(figsize=(12, 6))
for brand in top5:
    subset = df[df['Brand'] == brand]['Price_INR']
    sns.kdeplot(subset, ax=ax, label=brand, fill=False, linewidth=2)
ax.set_title('Price Distribution KDE by Brand (Top 5)', fontsize=13, fontweight='bold')
ax.set_xlabel('Price (INR)')
ax.legend()
plt.tight_layout()
plt.show()
print("Interpretation: Different brands have distinct price peaks showing their target market segments.")

In [ ]:
# Scatter — Weight vs Price
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df['Weight_KG'], df['Price_INR'], alpha=0.4, color='#9b59b6', s=30)
ax.set_title('Weight vs Price', fontsize=13, fontweight='bold')
ax.set_xlabel('Weight (KG)')
ax.set_ylabel('Price (INR)')
z = np.polyfit(df['Weight_KG'].dropna(),
               df.loc[df['Weight_KG'].notna(), 'Price_INR'], 1)
xs = np.linspace(df['Weight_KG'].min(), df['Weight_KG'].max(), 200)
ax.plot(xs, np.poly1d(z)(xs), 'r--', lw=2, label='Trend Line')
ax.legend()
plt.tight_layout()
plt.show()
print("Interpretation: Heavier bikes tend to be more expensive — indicating larger, premium engine bikes weigh more.")

---
## Step 9 — Conclusion

### 🔑 Key Findings

| Finding | Insight |
|---|---|
| **Price Range** | ₹50,000 (budget commuters) to ₹30L+ (superbikes) |
| **Most Models** | Kawasaki, Honda, Yamaha have the most variants |
| **Affordable Brands** | Hero, Bajaj, TVS — dominate budget segment |
| **Premium Brands** | BMW, Ducati, Kawasaki — luxury segment |
| **Engine CC & Price** | Strong positive correlation (r≈0.85) |
| **Mileage & Price** | Negative correlation — expensive bikes give less mileage |
| **Weight & Price** | Positive correlation — heavier bikes cost more |

### 📌 Answer to Problem Statement
> Engine displacement (CC) and brand are the strongest factors affecting bike prices in India.
> Commuter bikes (100-125cc) are priced affordably for the mass market, while performance bikes (300cc+) target the premium segment.
> Electric bikes (Ather, Ola) are competitively priced against 150-200cc petrol bikes — indicating growing EV market competition.
